# airllm-bench — GPU Reference Run (B4)

Google Colab notebook for the **GPU baseline** backend. Runs the same model,
prompt, and token budget as the local CPU/AirLLM benchmarks on a **T4 GPU**.

## Setup

1. **Runtime → Change runtime type → Hardware accelerator: T4 GPU**
2. Run all cells top-to-bottom.
3. Download `run_gpu_*.json` from the final cell into `results/` in the repo.

Benchmark parameters mirror `.env-example`: `Qwen/Qwen2.5-3B-Instruct`,
`PROMPT`, `MAX_NEW_TOKENS=16`, `SEED=42`.

In [ ]:
# Install dependencies matching airllm-bench pyproject.toml (CUDA torch on Colab).
!pip install -q \
    "torch>=2.2.0" \
    "transformers>=4.40.0" \
    "huggingface-hub>=0.23.0" \
    "safetensors>=0.4.3" \
    "accelerate>=0.30.0" \
    "psutil>=5.9.8"

In [ ]:
import json
import platform
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import psutil
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# ── Benchmark parameters (identical to local .env-example) ─────────────────────
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
PROMPT = "Explain virtual memory in one sentence."
MAX_NEW_TOKENS = 16
SEED = 42
AIRLLM_BENCH_VERSION = "0.1.0"

In [ ]:
assert torch.cuda.is_available(), "CUDA not available — select T4 GPU runtime."
device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
props = torch.cuda.get_device_properties(0)
print(f"Device: {device} ({gpu_name})")
print(f"VRAM: {props.total_memory / (1024 ** 2):.0f} MB")

In [ ]:
MB = 1024.0 * 1024.0


def capture_host_spec() -> dict:
    """Colab host metadata — same keys as airllm_bench.services.host_spec."""
    mem = psutil.virtual_memory()
    return {
        "cpu": platform.processor() or platform.machine(),
        "cpu_count_logical": psutil.cpu_count(logical=True),
        "cpu_count_physical": psutil.cpu_count(logical=False),
        "total_ram_mb": round(mem.total / MB, 1),
        "free_ram_mb": round(mem.available / MB, 1),
        "os": f"{platform.system()} {platform.release()}",
        "python": sys.version,
        "has_cuda": True,
        "device": f"cuda:{gpu_name}",
        "gpu_name": gpu_name,
        "gpu_total_memory_mb": round(props.total_memory / MB, 1),
    }

In [ ]:
def run_gpu_benchmark() -> dict:
    """Load fp16 model on CUDA, generate, and return a RunResult-compatible dict."""
    host = capture_host_spec()
    process = psutil.Process()
    peak_rss = process.memory_info().rss
    peak_system = psutil.virtual_memory().used

    torch.cuda.reset_peak_memory_stats(device)
    torch.manual_seed(SEED)

    # ── Load ──────────────────────────────────────────────────────────────────
    t0 = time.perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map={"": device},
    )
    model.eval()
    load_time_s = time.perf_counter() - t0
    peak_rss = max(peak_rss, process.memory_info().rss)
    peak_system = max(peak_system, psutil.virtual_memory().used)

    inputs = tokenizer(PROMPT, return_tensors="pt").to(device)

    # ── TTFT (prefill forward pass) ─────────────────────────────────────────
    t_prefill = time.perf_counter()
    with torch.no_grad():
        _ = model(**inputs)
    ttft_s = time.perf_counter() - t_prefill

    # ── Generate (greedy, seeded — matches TransformersCpuBackend) ──────────
    t1 = time.perf_counter()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
        )
    generate_time_s = time.perf_counter() - t1
    peak_rss = max(peak_rss, process.memory_info().rss)
    peak_system = max(peak_system, psutil.virtual_memory().used)

    new_ids = output_ids[0][inputs["input_ids"].shape[-1] :]
    output_text = tokenizer.decode(new_ids, skip_special_tokens=True)

    peak_gpu_mb = round(torch.cuda.max_memory_allocated(device) / MB, 1)
    tokens_per_s = MAX_NEW_TOKENS / generate_time_s if generate_time_s > 0 else None

    return {
        "backend": "gpu",
        "model_id": MODEL_ID,
        "prompt_chars": len(PROMPT),
        "max_new_tokens": MAX_NEW_TOKENS,
        "status": "success",
        "failure_reason": None,
        "load_time_s": round(load_time_s, 3),
        "ttft_s": round(ttft_s, 3),
        "generate_time_s": round(generate_time_s, 3),
        "total_runtime_s": round(load_time_s + generate_time_s, 3),
        "tokens_per_s": round(tokens_per_s, 5) if tokens_per_s else None,
        "peak_process_rss_mb": round(peak_rss / MB, 1),
        "peak_system_used_mb": round(peak_system / MB, 1),
        "peak_gpu_memory_mb": peak_gpu_mb,
        "output_preview": output_text[:200],
        "device": "cuda:T4",
        "host": host,
        "timestamp": datetime.now(tz=timezone.utc).isoformat(),
        "airllm_bench_version": AIRLLM_BENCH_VERSION,
    }

In [ ]:
result = run_gpu_benchmark()

ts_compact = datetime.now(tz=timezone.utc).strftime("%Y%m%d%H%M%S")
out_path = Path(f"run_gpu_{ts_compact}.json")
out_path.write_text(json.dumps(result, indent=2), encoding="utf-8")

print(f"Wrote {out_path}")
print(f"status={result['status']}  total={result['total_runtime_s']}s")
print(f"tokens/s={result['tokens_per_s']}  peak_gpu={result['peak_gpu_memory_mb']} MB")
print(f"output: {result['output_preview']}")

In [ ]:
from google.colab import files

files.download(str(out_path))